# ⚡ Kilat: General Pre-Training & 6-Gate Pre-Flight Assurance

```text
  ██╗  ██╗██╗██╗      █████╗ ████████╗
  ██║ ██╔╝██║██║     ██╔══██╗╚══██╔══╝
  █████╔╝ ██║██║     ███████║   ██║   
  ██╔═██╗ ██║██║     ██╔══██║   ██║   
  ██║  ██╗██║███████╗██║  ██║   ██║   
  ╚═╝  ╚═╝╚═╝╚══════╝╚═╝  ╚═╝   ╚═╝   
  ────────────────────────────────────────────────────────────
  ⚡ KILAT | Kernelized Lightweight Attention Transformer (v2.0)
  Architecture: Hybrid Sub-Quadratic Global Decay + Latent MLA
  ────────────────────────────────────────────────────────────
```

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Airukua/Kilat/blob/main/notebooks/kilat_general_pretraining_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Airukua%2FKilat-181717?style=flat-square&logo=github)](https://github.com/Airukua/Kilat)

Welcome to the **Kilat General Pre-Training** guide!
Kilat enforces the **Zero-Wasted-Compute Guarantee**: before burning thousands of GPU hours, every architecture and configuration must pass the **6-Gate Pre-Flight Verification Suite**.

In this notebook, you will:
1. Setup and audit the GPU environment.
2. Run Kilat's 6 pre-flight assurance gates (`Gate 0` through `Gate 6`).
3. Configure a sub-quadratic dense hybrid language model.
4. Execute pre-training with causal masking and numerical sanity assertions.
5. Export and verify the resulting checkpoint.

## 1. Environment Setup & Kilat Installation

In [ ]:
!rm -rf Kilat
!git clone https://github.com/Airukua/Kilat.git
%cd Kilat
!pip install -q -e ".[dev,integrations]"

from kilat.utils.banner import print_kilat_banner
print_kilat_banner()

## 2. The 6-Gate Pre-Flight Assurance Protocol
Before pre-training, Kilat runs a rigorous battery of invariant tests:
- **Gate 0 (Security)**: OWASP LLM/ML Top 10 + NCSC AST scan (no untrusted dynamic code execution).
- **Gate 1 (Config)**: Structural schema validation & hardware memory feasibility.
- **Gate 2 (Numerics)**: Dtype bounds, float16/bfloat16 underflow/overflow bounds.
- **Gate 3 (Causal)**: Strict autoregressive target masking (`labels = input_ids[1:]`, no future sequence leakage).
- **Gate 4 (Convergence)**: Overfit-one-batch proof (model must drive training loss down rapidly).
- **Gate 5 (Determinism)**: Resuming training from step $N$ produces exact bitwise loss and RNG parity.
- **Gate 6 (Hardware)**: Throughput telemetry and VRAM profiling.

In [ ]:
# Run full Pre-Flight Assurance on the debug configuration
!kilat verify --config configs/laptop_cpu.yaml --output-dir verify_signoff

## 3. Configure Dense Sub-Quadratic Model
Kilat's Macro-Synergy architecture couples **Global Decay ($O(1)$ linear recurrent scan)** with **Latent MLA** for dense efficiency.

In [ ]:
import torch
from kilat.configs.model_config import KilatConfig
from kilat.models.transformer import KilatTransformer

config = KilatConfig(
    vocab_size=50257,
    d_model=384,
    num_layers=8,
    num_heads=6,
    latent_dim=96,
    max_position_embeddings=2048,
    ffn_mode="dense",
    use_bias=False,
    dropout=0.0
)

model = KilatTransformer(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"⚡ Dense Model Initialized with {total_params / 1e6:.2f}M parameters")

## 4. Run Pre-Training Loop
Execute training with synthetic or tokenized data using `KilatTrainer`.

In [ ]:
from kilat.engine.args import TrainingArguments
from kilat.engine.trainer import KilatTrainer
from kilat.data.dataset import PretrainingDataset
from kilat.data.collator import KilatDataCollator

# Configure training parameters
args = TrainingArguments(
    output_dir="./checkpoints/dense_general",
    learning_rate=8e-4,
    warmup_steps=100,
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    mixed_precision="fp16" if not torch.cuda.is_bf16_supported() else "bf16",
    save_steps=250,
    logging_steps=25,
    seed=42
)

print("⚡ Training configured for:", args.max_steps, "steps")

## 5. Active Chaos Engineering Verification
Ensure that your training harness survives active fault injection by running Kilat's **Saboteur** suite.

In [ ]:
# Test active chaos resilience (C-01 to C-08)
!kilat chaos --scenarios all